# Raport kontrolny okresu

Przegląd jednego przygotowanego okresu przed ustawieniem `checked = true` w katalogu. Okres wybiera zmienna `PERIOD` niżej (albo zmienna środowiskowa `PERIOD` przy `nbconvert --execute`). Dane: `data/prepared/<obiekt>/<id>.db` z `uv run python -m dms_analysis.data prepare`; ten sam okres w Grafanie: dashboard „Okresy”.

Wykresy są statyczne: całe okresy pokazują najniższy i najwyższy odczyt w każdym z 3000 przedziałów (spadki i skoki zostają widoczne), pełna rozdzielczość tylko w podglądzie zdarzeń i doby (sekcje 3, 4 i 7).

Po przeglądzie: `checked = true` w pliku obiektu w `catalog/`, potem `prepare`, żeby status trafił do baz.

In [ ]:
import os
from IPython.display import Markdown, display

from dms_analysis.data import check
from dms_analysis.data.catalog import load_catalog

PERIOD = os.environ.get("PERIOD", "swietochlowice-1/0004-2026-10-30")

catalog = load_catalog()
period = check.load_period(PERIOD)
warning = check.staleness_warning(catalog, PERIOD)

## 1. Opis okresu

In [ ]:
if warning:
    display(Markdown(f"<span style='color:#c0392b'><b>Uwaga:</b> {warning}</span>"))
display(Markdown(f"**{period.title}** · {check.time_description(period.info)}"))
check.description_table(period)

## 2. Cały okres

Wskazania, pod nimi sygnały surowe i oczyszczone (obecny wariant reguł, zdarzenia wpisane ręcznie zaznaczone na żółto), na dole temperatury. Problemy z katalogu (`[[period.issue]]`) zaznaczone pomarańczowym pasmem.

In [ ]:
check.overview_figure(period).show()

## 3. Zdarzenia a wskazania

Przyjęte odsypy bez spadku wskazania i zasypy bez wzrostu: zmiana = mediana z 3 s po końcu sygnału − mediana z 3 s przed początkiem. Podgląd wybranego: numer wiersza w `SUSPICIOUS`.

In [ ]:
suspicious = check.suspicious_events(period)
display(Markdown(f"{len(suspicious)} zdarzeń bez oczekiwanej zmiany wskazania"))
suspicious.reset_index(drop=True)

In [ ]:
SUSPICIOUS = 0

if len(suspicious):
    check.suspicious_figure(period, SUSPICIOUS).show()

## 4. Ręczne zdarzenia

Każde zdarzenie wpisane ręcznie (`[[period.manual_event]]`) na tle wskazań, ±15 min.

In [ ]:
figures = check.manual_event_figures(period)
display(Markdown(f"{len(figures)} ręcznych zdarzeń"))
for figure in figures:
    figure.show()

## 5. Usunięte odczyty temperatur

Kody błędów DS18B20 (85, −127), ucięte adresy i czujniki spoza konfiguracji sprzętu.

In [ ]:
removed = check.removed_summary(period)
display(removed if len(removed) else Markdown("nic nie usunięto"))
if len(period.removed):
    check.removed_figure(period).show()

## 6. Luki, części i pełne doby

Część kończy się na przerwie we wskazaniach dłuższej niż 60 s. Doba = doba kalendarzowa w czasie `Europe/Warsaw`; pełna, gdy cała leży w jednej części (doba zmiany czasu ma 23 albo 25 h). Pełna doba, w którą wchodzi problem z katalogu (`[[period.issue]]`, np. zakłócenia wskazań), jest oznaczona osobno („pełna z problemem”, pomarańczowa); co z nią robić w miarach dobowych — decyzja później. Przy czasie zegara urządzenia dób nie wyznaczamy.

In [ ]:
check.parts_table(period)

In [ ]:
days = check.period_days(period)
display(check.days_table(period))
check.days_figure(period).show()

## 7. Podgląd doby

Pełna rozdzielczość jednej doby; `DAY` = data (domyślnie pierwsza pełna doba).

In [ ]:
DAY = days[0] if days else period.start.date()

check.day_figure(period, DAY).show()